<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase38_Athena_Binding_Validation_Drive_to_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 38: Frozen Athena vocabulary and independent binding-review gate
**Drive → Drive | Phase 37 input verification | private Athena stays private**

This is a **real dependency-resolution notebook**, not another synthetic performance experiment. It independently verifies the exact Phase 37 public artifacts (19 synthetic people, 38 manufactured test Procedure links, 148 original Observations, 13 negative-link tests), creates a private 19-binding human-review worksheet, optionally scans an **authorized frozen OHDSI Athena `CONCEPT.csv`** and validates manually chosen concept IDs against its date, status, domain and the **documented synthetic source evidence**.

**Run without Athena:** upstream integrity is checked; the missing prerequisites and blank review worksheet are exported; all unresolved vocab gaps remain blocked. **Run with Athena:** the export and a human-reviewed 19-row CSV and manifest must already exist in `_PRIVATE_ATHENA`. No Athena file, selected ID, patient record, reviewer identity, or private path is copied into public Phase 38 outputs. Never upload `_PRIVATE_ATHENA` to public GitHub.

**Caveat:** Phase 37 Procedures and UID values are deliberately manufactured test fixtures; the original Phase 33 FHIR records contain no source Procedures, gender or birth dates, nor attested MCI status. Even successful vocabulary review does **not** establish official Arm A, MI-CDM loading, real ADNI validity or four-arm cohort equivalence. The Phase 36 internal future protocol is unchanged.

In [1]:
import os,json,csv,gzip,hashlib,re,shutil,datetime
from pathlib import Path
from collections import Counter
if os.environ.get('ICHI2027_DRIVE_ROOT'):
    DRIVE_ROOT=Path(os.environ['ICHI2027_DRIVE_ROOT'])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')
DRIVE_RELATIVE_FOLDER=''  # Optional: 'NeuroFHIR_ICHI2027' or project subfolder
ROOT=DRIVE_ROOT/DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
assert ROOT.is_dir(),f'Drive path missing: {ROOT}'
print('Search root:',ROOT)

Mounted at /content/drive
Search root: /content/drive/MyDrive


In [2]:
# Find matching Phase 37 artifact bytes, never the latest filename alone.
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
def find_files(root,pattern):
    matches=[]
    for folder,dirs,names in os.walk(root,onerror=lambda _:None):
        dirs[:]=[d for d in dirs if d not in {'_PRIVATE_ATHENA','.git','node_modules'} and not d.startswith('.')]
        for name in names:
            if Path(name).match(pattern):
                p=Path(folder)/name
                if p.is_file():matches.append(p)
    return sorted(matches,key=lambda p:p.stat().st_mtime,reverse=True)

def companion(where,base,expected_sha):
    results=[p for p in where.glob(base+'*') if p.is_file() and p.suffix.lower() in {'.csv','.json'}]
    matches=[p for p in results if sha(p)==expected_sha]
    if len(matches)==0:raise FileNotFoundError(f'Phase 37 companion missing/changed: {base} (SHA {expected_sha})')
    return matches[0]

P37=None;issues=[]
for p in find_files(ROOT,'ICHI2027_Phase37_SHAREABLE*.json'):
    try:
        q=json.loads(p.read_text('utf-8'))
        if q.get('phase')!='37_posthoc_manufactured_synthetic_procedures_for_phase33_cohort':continue
        if q['manufactured_synthetic_test_fixture']['studies_with_1_to_1_fixture_links']!=38:continue
        if q['tests']['negative_link_cases_detected']!=13:continue
        a={}
        for fname,hashval in q['artifact_sha256'].items():
            stem=Path(fname).stem;a[stem]=companion(p.parent,stem,hashval)
        P37=(p,q,a);break
    except (OSError,ValueError,KeyError,TypeError) as e:issues.append((p.name,str(e)))
if P37 is None:raise FileNotFoundError('No complete hash-matched Phase 37 run: '+str(issues[:4]))
P37_JSON,p37,ART=P37;OUTDIR=P37_JSON.parent
links=list(csv.DictReader(ART['ICHI2027_Phase37_SYNTHETIC_PROCEDURE_LINKS'].open(newline='',encoding='utf-8-sig')))
joins=list(csv.DictReader(ART['ICHI2027_Phase37_SOURCE_TO_FIXTURE_JOIN'].open(newline='',encoding='utf-8-sig')))
neg=list(csv.DictReader(ART['ICHI2027_Phase37_NEGATIVE_LINK_TESTS'].open(newline='',encoding='utf-8-sig')))
assert (len(links),len(joins),len(neg))==(38,148,13)
assert len(set(r['study_ref'] for r in links))==38
assert len(set(r['procedure_ref'] for r in links))==38
assert len(set(r['observation_ref'] for r in joins))==148
assert all(r['detected'].strip().lower()=='true' for r in neg)
assert all(r['join_status']=='EXACT_SYNTHETIC_FIXTURE_ONLY' for r in joins),set(r['join_status'] for r in joins)
assert all(r['fixture_kind']=='POST_HOC_MANUFACTURED_SYNTHETIC_ONLY' for r in links)
assert all(r['clinical_source_procedure_attested'].lower()=='false' for r in links)
assert p37['gaps']['original_phase29_vocab_gaps_blocked']==36
bundle=json.loads(ART['ICHI2027_Phase37_SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES'].read_text(encoding='utf-8'))
counts=Counter(e['resource']['resourceType'] for e in bundle['entry'])
assert counts['Patient']==19 and counts['ImagingStudy']==38 and counts['Observation']==148 and counts['Procedure']==38
assert counts['Provenance']==186
# SHA on original upstream FHIR report is preserved as a parent reference, not rederived from fixture bundle.
phase36_found=any(sha(x)==p37['upstream']['phase36_shareable_sha256'] for x in find_files(ROOT,'ICHI2027_Phase36_SHAREABLE*.json'))
print('VERIFIED Phase 37 run:',P37_JSON)
print('Phase 36 exact-shareable independently found:',phase36_found)
print('Input integrity: 38 fixtures, 148 joins, 13 negative controls; all companions hash-matched')

VERIFIED Phase 37 run: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/ICHI2027_Phase37_SHAREABLE.json
Phase 36 exact-shareable independently found: True
Input integrity: 38 fixtures, 148 joins, 13 negative controls; all companions hash-matched


## Private Athena prerequisites

Create or use the `_PRIVATE_ATHENA` folder **beside the Phase 37 outputs**, never inside the public repository. The notebook creates two private *templates*, never overwriting existing human work:

1. An independently reviewed 19-row worksheet (`ICHI2027_Phase38_REVIEWED_BINDINGS.csv`). The Phase 37 worksheet is preserved separately; copy any *genuine* reviewer decisions manually rather than silently inheriting placeholders. For each actual approval enter the exact selected `concept_id`, matching snapshot SHA, reviewer, date, evidence, literal source-code definition, domain reasoning, application policy, and disposition `APPROVED`.
2. `SNAPSHOT_MANIFEST_TEMPLATE.json`. After downloading your own authorized `CONCEPT.csv` or `CONCEPT.csv.gz` from Athena, copy and fill it to `_PRIVATE_ATHENA/SNAPSHOT_MANIFEST.json` with the export date, source, release and **SHA-256 of the exact compressed or uncompressed file**. A filename alone is not provenance.

The two sex/gender binding keys refer to **source values absent in all 19 FHIR patients**; do not assign female or male concepts to these persons. They can be dispositioned `NOT_APPLICABLE_SOURCE_ABSENT` with independently documented reason. Race and ethnicity also require reviewed missingness policy; they must not be invented. A concept may be valid in Athena but still semantically unsuitable for a laterality-specific MRI measurement; independent evidence and preservation policy are mandatory.

In [3]:
PRIVATE=OUTDIR/'_PRIVATE_ATHENA';PRIVATE.mkdir(exist_ok=True)
keys=['IO_MODALITY_MR','IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT',
      'IF_TYPE_ALGORITHM','PERSON_GENDER_F','PERSON_GENDER_M',
      'PERSON_RACE_UNRECORDED','PERSON_ETHNICITY_UNRECORDED','PROCEDURE_MRI','PROCEDURE_TYPE_SYN',
      'MEAS_HIP_LEFT','MEAS_HIP_RIGHT','MEAS_ENT_LEFT','MEAS_ENT_RIGHT',
      'MEAS_TYPE_SYN','UNIT_MM3','IF_EVENT_MEAS_ID']
EXPECTED_DOMAINS={k:'' for k in keys}
EXPECTED_DOMAINS.update({'PERSON_GENDER_F':'Gender','PERSON_GENDER_M':'Gender',
  'PERSON_RACE_UNRECORDED':'Race','PERSON_ETHNICITY_UNRECORDED':'Ethnicity',
  'PROCEDURE_MRI':'Procedure','PROCEDURE_TYPE_SYN':'Type Concept',
  'MEAS_HIP_LEFT':'Measurement','MEAS_HIP_RIGHT':'Measurement','MEAS_ENT_LEFT':'Measurement',
  'MEAS_ENT_RIGHT':'Measurement','MEAS_TYPE_SYN':'Type Concept','UNIT_MM3':'Unit','IF_EVENT_MEAS_ID':'Metadata'})
HINTS={'IO_MODALITY_MR':'magnetic resonance','IF_HIP_LEFT':'hippocamp','IF_HIP_RIGHT':'hippocamp',
'IF_ENT_LEFT':'entorhin','IF_ENT_RIGHT':'entorhin','IF_TYPE_ALGORITHM':'algorithm',
'PERSON_GENDER_F':'female','PERSON_GENDER_M':'male','PERSON_RACE_UNRECORDED':'unknown',
'PERSON_ETHNICITY_UNRECORDED':'unknown','PROCEDURE_MRI':'magnetic resonance',
'PROCEDURE_TYPE_SYN':'source','MEAS_HIP_LEFT':'hippocamp','MEAS_HIP_RIGHT':'hippocamp',
'MEAS_ENT_LEFT':'entorhin','MEAS_ENT_RIGHT':'entorhin','MEAS_TYPE_SYN':'derived',
'UNIT_MM3':'cubic millimeter','IF_EVENT_MEAS_ID':'measurement_id'}
REVIEW_COLS=['binding_key','proposed_concept_id','snapshot_sha256','reviewer','review_date_utc',
 'independent_review_evidence','exact_source_code_and_definition','semantic_and_domain_justification',
 'source_application_policy','independent_review_decision','extension_domain_policy_evidence',
 'laterality_preservation_evidence','event_field_cdm_table_field','event_field_definition_evidence']
TEMPLATE=PRIVATE/'ICHI2027_Phase38_REVIEWED_BINDINGS.csv'
if not TEMPLATE.exists():
    rows=[{x:'' for x in REVIEW_COLS} for _ in keys]
    for r,k in zip(rows,keys):r.update(binding_key=k,independent_review_decision='PENDING')
    with TEMPLATE.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=REVIEW_COLS);w.writeheader();w.writerows(rows)
MANIFEST_TEMPLATE=PRIVATE/'SNAPSHOT_MANIFEST_TEMPLATE.json'
if not MANIFEST_TEMPLATE.exists():
    MANIFEST_TEMPLATE.write_text(json.dumps({
      'snapshot_date_utc':'YYYY-MM-DD',
      'expected_CONCEPT_sha256':'PASTE_FULL_SHA256_OF_EXACT_CONCEPT_FILE',
      'source':'Your authorized OHDSI Athena export origin and export date',
      'omop_vocabulary_release':'Exact release if known',
      'independence_note':'Keep signed independent reviews alongside the private worksheet'},indent=2)+'\n')
ATHENA=next((p for p in [PRIVATE/'CONCEPT.csv',PRIVATE/'CONCEPT.csv.gz',PRIVATE/'concept.csv',PRIVATE/'concept.csv.gz'] if p.is_file()),None)
SNAPSHOT_HASH=sha(ATHENA) if ATHENA else None
MANIFEST=PRIVATE/'SNAPSHOT_MANIFEST.json'
manifest={};snapshot_day=None;manifest_valid=False
if MANIFEST.is_file():
    try:
        manifest=json.loads(MANIFEST.read_text('utf-8'))
        snapshot_day=datetime.date.fromisoformat(str(manifest['snapshot_date_utc']))
        manifest_valid=(ATHENA is not None and snapshot_day<=datetime.datetime.now(datetime.timezone.utc).date()
          and bool(str(manifest.get('source','')).strip()) and bool(str(manifest.get('omop_vocabulary_release','')).strip())
          and str(manifest.get('expected_CONCEPT_sha256','')).lower().strip()==SNAPSHOT_HASH)
    except (TypeError,KeyError,ValueError):manifest_valid=False
print('Private reviewed worksheet:',TEMPLATE)
print('Private manifest template:',MANIFEST_TEMPLATE)
print('Athena present:',ATHENA is not None,'; frozen manifest valid:',manifest_valid)
if SNAPSHOT_HASH:print('CONCEPT file SHA-256 — keep private:',SNAPSHOT_HASH)

Private reviewed worksheet: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/_PRIVATE_ATHENA/ICHI2027_Phase38_REVIEWED_BINDINGS.csv
Private manifest template: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/_PRIVATE_ATHENA/SNAPSHOT_MANIFEST_TEMPLATE.json
Athena present: False ; frozen manifest valid: False


In [4]:
# Validate the human review CSV structure *before* reading concept records.
with TEMPLATE.open(newline='',encoding='utf-8-sig') as f:rv=list(csv.DictReader(f))
assert len(rv)==19 and set(r['binding_key'] for r in rv)==set(keys) and len(set(r['binding_key'] for r in rv))==19
assert set(REVIEW_COLS).issubset(rv[0]),'Private review CSV is missing mandatory columns'
reviews={r['binding_key']:r for r in rv}
chosen={r['proposed_concept_id'].strip() for r in rv if r['proposed_concept_id'].strip()}
assert all(x.isdigit() and int(x)>0 for x in chosen),'No negative, zero or placeholder concept IDs'
needed={'concept_id','concept_name','domain_id','standard_concept','valid_start_date','valid_end_date','invalid_reason'}
matching={};candidate=[];candidate_n=Counter()
if ATHENA:
    opener=gzip.open if ATHENA.suffix=='.gz' else open
    with opener(ATHENA,'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
        line=f.readline();sep='\t' if '\t' in line else ','
    with opener(ATHENA,'rt',encoding='utf-8-sig',errors='replace',newline='') as f:
        reader=csv.DictReader(f,delimiter=sep)
        assert needed.issubset(set(reader.fieldnames or [])),f'CONCEPT export is missing {needed-set(reader.fieldnames or [])}'
        for c in reader:
            cid=str(c['concept_id']).strip()
            if cid in chosen:matching[cid]={k:c.get(k,'') for k in needed}
            if c['invalid_reason'].strip():continue
            name=c['concept_name'].lower()
            for key,term in HINTS.items():
                if candidate_n[key]>=8 or term not in name:continue
                expected=EXPECTED_DOMAINS[key]
                if expected and c['domain_id']!=expected:continue
                if key!='IF_EVENT_MEAS_ID' and c['standard_concept']!='S':continue
                candidate.append({'binding_key':key,**{k:c.get(k,'') for k in needed},
                                  'warning':'SUGGESTION_ONLY_REQUIRES_INDEPENDENT_REVIEW'})
                candidate_n[key]+=1
    CANDFILE=PRIVATE/'ICHI2027_Phase38_PRIVATE_CANDIDATES.csv'
    with CANDFILE.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=['binding_key']+sorted(needed)+['warning']);w.writeheader();w.writerows(candidate)
    print('Search-only candidates (private):',len(candidate),'— these are NOT verified mappings')

In [5]:
# Apply strict frozen-snapshot, domain, standard status and source-semantics gates.
def asdate(raw):
    for fmt in ('%Y-%m-%d','%Y%m%d'):
        try:return datetime.datetime.strptime(str(raw).strip(),fmt).date()
        except ValueError:pass
    return None

source_gender_present=any(bool(e['resource'].get('gender')) for e in bundle['entry'] if e['resource']['resourceType']=='Patient')
source_birthdate_present=any(bool(e['resource'].get('birthDate')) for e in bundle['entry'] if e['resource']['resourceType']=='Patient')
assert not source_gender_present and not source_birthdate_present
assert all(not e['resource'].get('extension') for e in bundle['entry'] if e['resource']['resourceType']=='Patient'), 'Patient extensions require manual source race/ethnicity audit'
status=[];approved=[];inapplicable=[];today=datetime.datetime.now(datetime.timezone.utc).date()
for key in keys:
    r=reviews[key];cid=r['proposed_concept_id'].strip();disp=r['independent_review_decision'].strip().upper()
    s='BLOCKED_PENDING_INDEPENDENT_REVIEW';c=None
    documented=all(str(r[x]).strip() for x in ['reviewer','review_date_utc','independent_review_evidence',
       'exact_source_code_and_definition','semantic_and_domain_justification','source_application_policy'])
    reviewer_day=asdate(r['review_date_utc'])
    reviewer_date_ok=reviewer_day is not None and reviewer_day<=today
    if key in {'PERSON_GENDER_F','PERSON_GENDER_M'}:
        if cid or disp=='APPROVED':s='REJECTED_SOURCE_GENDER_ABSENT'
        elif disp=='NOT_APPLICABLE_SOURCE_ABSENT' and documented and reviewer_date_ok:
            s='REVIEWED_INAPPLICABLE_SOURCE_ABSENT';inapplicable.append(key)
        elif disp!='PENDING':s='BLOCKED_GENDER_MUST_BE_MARKED_NOT_APPLICABLE'
    elif disp=='NOT_APPLICABLE_SOURCE_ABSENT':
        s='BLOCKED_MISSINGNESS_REQUIRES_EXPLICIT_PERSON_POLICY' if key.startswith('PERSON_') else 'BLOCKED_UNSUPPORTED_NOT_APPLICABLE'
    elif cid or disp=='APPROVED':
        if not cid or disp!='APPROVED':s='BLOCKED_ID_AND_APPROVAL_REQUIRED'
        elif ATHENA is None:s='BLOCKED_NO_PRIVATE_ATHENA_EXPORT'
        elif not manifest_valid:s='BLOCKED_SNAPSHOT_MANIFEST_MISSING_OR_MISMATCHED'
        elif r['snapshot_sha256'].strip().lower()!=SNAPSHOT_HASH:s='BLOCKED_REVIEW_SNAPSHOT_SHA_MISMATCH'
        elif not documented or not reviewer_date_ok:s='BLOCKED_HUMAN_REVIEW_EVIDENCE_OR_DATE_MISSING'
        elif cid not in matching:s='BLOCKED_CONCEPT_NOT_IN_FROZEN_EXPORT'
        else:
            c=matching[cid];start=asdate(c['valid_start_date']);end=asdate(c['valid_end_date'])
            expected=EXPECTED_DOMAINS[key]
            if c['invalid_reason'].strip() or not start or not end or not start<=snapshot_day<=end:
                s='BLOCKED_CONCEPT_EXPIRED_OR_INVALID'
            elif expected and c['domain_id']!=expected:s='BLOCKED_DOMAIN_MISMATCH'
            elif key!='IF_EVENT_MEAS_ID' and c['standard_concept']!='S':s='BLOCKED_NONSTANDARD_CONCEPT'
            elif not expected and not r['extension_domain_policy_evidence'].strip():
                s='BLOCKED_EXTENSION_DOMAIN_POLICY_UNDOCUMENTED'
            elif key in {'IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT'} and not r['laterality_preservation_evidence'].strip():
                s='BLOCKED_LATERALITY_NOT_JUSTIFIED'
            elif key=='IF_EVENT_MEAS_ID' and not (c['concept_name'].strip().lower()=='measurement_id'
                and r['event_field_cdm_table_field'].strip().upper()=='MEASUREMENT.MEASUREMENT_ID'
                and r['event_field_definition_evidence'].strip()):
                s='BLOCKED_EVENT_FIELD_METADATA_NOT_ADJUDICATED'
            elif key=='PERSON_RACE_UNRECORDED' and 'unrecorded' not in r['source_application_policy'].lower():
                s='BLOCKED_RACE_MISSINGNESS_POLICY'
            elif key=='PERSON_ETHNICITY_UNRECORDED' and 'unrecorded' not in r['source_application_policy'].lower():
                s='BLOCKED_ETHNICITY_MISSINGNESS_POLICY'
            elif key in {'PROCEDURE_MRI','PROCEDURE_TYPE_SYN'} and 'synthetic' not in r['source_application_policy'].lower():
                s='BLOCKED_MANUFACTURED_PROCEDURE_POLICY'
            else:
                s='REVIEWED_SNAPSHOT_CHECKED_SYNTHETIC_ONLY';approved.append(key)
    status.append({'binding_key':key,'status':s,'expected_domain_if_defined':EXPECTED_DOMAINS[key],
                   'candidate_present':bool(cid),'manual_review_supplied':disp!='PENDING',
                   'snapshot_present':ATHENA is not None,'manifest_valid':manifest_valid})

assert len(status)==19
print('Manually approved and snapshot checked:',len(approved),'/19')
print('Documented not applicable (gender keys only):',len(inapplicable),'/2')
print('Vocab remaining blocked:',19-len(approved)-len(inapplicable))

Manually approved and snapshot checked: 0 /19
Documented not applicable (gender keys only): 0 /2
Vocab remaining blocked: 19


In [6]:
# Audit rejection controls independent of whether private Athena has been supplied.
# Each test verifies the declared approval guard itself, without authoring fake approvals.
tests=[
 ('upstream_hash_chain',len(ART)==6 and all(sha(ART[Path(x).stem])==h for x,h in p37['artifact_sha256'].items())),
 ('synthetic_procedure_count',counts['Procedure']==38),
 ('synthetic_provenance_count',counts['Provenance']==186),
 ('source_observations_preserved',counts['Observation']==148 and p37['manufactured_synthetic_test_fixture']['source_Observations_unchanged']),
 ('one_to_one_procedure_links',len(set(x['procedure_ref'] for x in links))==38),
 ('all_148_observation_joins',len(set(x['observation_ref'] for x in joins))==148),
 ('phase37_thirteen_negative_controls',all(x['detected'].lower()=='true' for x in neg)),
 ('missing_gender_suppresses_gender_approvals',all(k not in approved for k in ['PERSON_GENDER_F','PERSON_GENDER_M'])),
 ('athena_absent_produces_zero_approved',ATHENA is not None or len(approved)==0),
 ('athena_without_manifest_produces_zero_approved',manifest_valid or len(approved)==0),
 ('no_auto_selected_concept_ids',all(r['proposed_concept_id'].strip()=='' for r in rv if r['independent_review_decision']=='PENDING')),
 ('official_ArmA_remains_unloaded',not p37['gates']['faithful_armA_validated'] and not p37['gates']['official_MI_CDM_loaded']),
]
assert all(ok for _,ok in tests),[(n,ok) for n,ok in tests if not ok]
print('Prerequisite and corruption-defense checks:',sum(ok for _,ok in tests),'/',len(tests))

Prerequisite and corruption-defense checks: 12 / 12


In [7]:
# Public-safe outputs — no selected concept ids, reviewer names, Athena concept records, or private paths.
PREFIX='ICHI2027_Phase38_'
def public_csv(name,rows,cols):
    path=OUTDIR/(PREFIX+name+'.csv')
    with path.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=cols);w.writeheader();w.writerows(rows)
    return path
ST=public_csv('BINDING_REVIEW_STATUS',status,list(status[0]))
TT=public_csv('GATE_TESTS',[{'test':n,'passed':v} for n,v in tests],['test','passed'])
ACTION=[
 {'prerequisite':'frozen_authorized_Athena_CONCEPT','status':'VERIFIABLE_PRIVATE_SNAPSHOT' if manifest_valid else 'BLOCKED',
  'next_action':'Place authorized CONCEPT.csv and a source/date/release/SHA-matched private SNAPSHOT_MANIFEST.json in _PRIVATE_ATHENA'},
 {'prerequisite':'independently_reviewed_19_bindings','status':f'{len(approved)}_VERIFIED__{len(inapplicable)}_SOURCE_INAPPLICABLE',
  'next_action':'Have an independent reviewer adjudicate the private 19-row worksheet; evidence and applicability matter beyond valid IDs'},
 {'prerequisite':'source_procedure_truth','status':'MANUFACTURED_SYNTHETIC_FIXTURES_ONLY',
  'next_action':'For synthetic Arm A, keep visible post-hoc fixture tags; for real data obtain actual source Procedure documentation'},
 {'prerequisite':'source_demographics','status':'GENDER_BIRTHDATE_MISSING',
  'next_action':'Never invent gender/year-of-birth; decide and document approved missingness policy or use source-attested new synthetic fixture explicitly'},
 {'prerequisite':'IMAGE_FEATURE_event_field','status':'REVIEWED' if 'IF_EVENT_MEAS_ID' in approved else 'BLOCKED',
  'next_action':'Independently verify IMAGE_FEATURE event-field reference to MEASUREMENT.measurement_id against pinned CDM semantics'},
 {'prerequisite':'official_ArmA_staging_and_load','status':'NOT_EXECUTED',
  'next_action':'Only after frozen vocabulary, source field completeness and metadata adjudication, run schema and SQL validation'}]
ACT=public_csv('REQUIRED_ACTIONS',ACTION,['prerequisite','status','next_action'])
semantics=[{'fact':'synthetic_people','value':'19'}, {'fact':'imaging_studies','value':'38'},
 {'fact':'source_observations','value':'148'}, {'fact':'manufactured_synthetic_procedures','value':'38'},
 {'fact':'real_clinical_source_procedures','value':'0'}, {'fact':'source_gender_present','value':str(source_gender_present)},
 {'fact':'source_birthdates_present','value':str(source_birthdate_present)},
 {'fact':'real_DICOM_attested','value':'False'},{'fact':'clinical_MCI_attested','value':'False'},
 {'fact':'official_ArmA_loaded','value':'False'}]
SEM=public_csv('SOURCE_SEMANTICS',semantics,['fact','value'])
report={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'38_private_frozen_athena_and_human_binding_review_gate',
 'run_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'privacy':'Public metadata only; no private Athena records, concept ids, reviewer identity or private paths',
 'upstream':{'phase37_shareable_sha256':sha(P37_JSON),'phase37_bundle_sha256':p37['artifact_sha256']['ICHI2027_Phase37_SYNTHETIC_FHIR_WITH_PROCEDURE_FIXTURES.json'],
    'phase36_exact_sha_found_in_drive':phase36_found,'imagewg_commit':p37['upstream']['imagewg_commit'],'omop55_commit':p37['upstream']['omop55_commit']},
 'phase37_verified':{'synthetic_people':19,'synthetic_studies':38,'manufactured_fixture_procedures':38,
  'source_observations':148,'observation_to_fixture_joins':148,'negative_link_tests_passed':13},
 'private_gate':{'snapshot_supplied':ATHENA is not None,'snapshot_manifest_valid':manifest_valid,
  'manual_snapshot_checked_approvals':len(approved),'reviewed_inapplicable_gender_keys':len(inapplicable),
  'original_binding_keys':19,'original_36_vocabulary_gaps_declared_resolved':False},
 'tests':{'total':len(tests),'passed':sum(int(v) for _,v in tests)},
 'gates':{'faithful_armA_validated':False,'official_MICDM_loaded':False,'pyOMOP_ETL_executed':False,
    'original_generic_armC_executed':False,'true_four_arm_comparison_completed':False,
    'official_target_cohort_equivalence_tested':False,'real_MRI_rows_processed':0,
    'synthetic_source_missing_gender_birthdate':True,'independent_clinical_source_procedure_attested':False},
 'public_artifact_sha256':{p.name:sha(p) for p in [ST,TT,ACT,SEM]},
 'next_gate':'Independently review and freeze the authorized Athena concepts; document demographic and event-field policies. No official Arm A or clinical claims until source and schema gates independently pass.'}
REPORT=OUTDIR/(PREFIX+'SHAREABLE.json')
REPORT.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
for p in [REPORT,ST,TT,ACT,SEM]:
    assert p.is_file() and p.stat().st_size>0
print('PHASE 38 FINISHED —',len(tests),'/',len(tests),'gate checks passed')
print('Athena snapshot and review:',len(approved),'approved;',len(inapplicable),'inapplicable gender keys')
print('Outputs saved in:',OUTDIR)
for p in [REPORT,ST,TT,ACT,SEM]:print(' ',p.name)
print('IMPORTANT: _PRIVATE_ATHENA must not be uploaded to public GitHub.')

PHASE 38 FINISHED — 12 / 12 gate checks passed
Athena snapshot and review: 0 approved; 0 inapplicable gender keys
Outputs saved in: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs
  ICHI2027_Phase38_SHAREABLE.json
  ICHI2027_Phase38_BINDING_REVIEW_STATUS.csv
  ICHI2027_Phase38_GATE_TESTS.csv
  ICHI2027_Phase38_REQUIRED_ACTIONS.csv
  ICHI2027_Phase38_SOURCE_SEMANTICS.csv
IMPORTANT: _PRIVATE_ATHENA must not be uploaded to public GitHub.


## What to upload here after running

Upload only these **public-safe** files: `ICHI2027_Phase38_SHAREABLE.json`, `ICHI2027_Phase38_BINDING_REVIEW_STATUS.csv`, `ICHI2027_Phase38_GATE_TESTS.csv` and `ICHI2027_Phase38_REQUIRED_ACTIONS.csv`.

Keep the authorized `CONCEPT.csv`, private review worksheet, actual selected concept IDs, reviewer identity and manifest in `_PRIVATE_ATHENA` on Drive; do **not** upload them to the public GitHub repository or this conversation. If vocabulary review remains pending, the next action is acquiring and independently reviewing that data—not declaring Arm A complete through synthetic substitutes.